# Giai đoạn 3: Huấn luyện Kiến trúc YOLOv11 (BDD100k)
Tài liệu này thực thi quy trình định chuẩn:
1. Khởi tạo và thiết lập môi trường thư viện YOLOv11 tiên tiến nhất.
2. Kiểm định tài nguyên phần cứng (Khả năng đáp ứng của GPU Cuda).
3. Kích hoạt kỹ thuật Tích lũy Gradient (Gradient Accumulation) nhằm vượt qua rào cản VRAM vật lý, kết hợp cơ chế kiểm soát Overfitting.

In [ ]:
# Khởi tạo thư viện cốt lõi
import sys
!{sys.executable} -m pip install -U ultralytics
!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# Kiểm định tính khả dụng của phần cứng GPU
import torch
if torch.cuda.is_available():
    print(f"Hệ thống nhận diện phần cứng xử lý: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Dung lượng VRAM khả dụng thực tế: {vram:.2f} GB")
else:
    print("Lỗi nghiêm trọng: Không phát hiện nhân tính toán CUDA. Quá trình phân tích sẽ bị gián đoạn.")

## Kích hoạt Mô hình YOLOv11 Nano (YOLO11n)
Áp dụng cấu hình phân rã khối lượng công việc nhằm bảo đảm an toàn VRAM (4GB):
- **Kích thước vật lý (Batch Size)** = 8 (Mức độ dung nạp tối đa của VRAM 4GB).
- **Hệ số giả lập Gradient tự động (Auto-Accumulate)**: Thư viện Ultralytics tự động thiết lập Accumulate = 64 / Batch. Với Batch=8, hệ thống sẽ ngầm định tích lũy ma trận sai số qua 8 chu kỳ trước khi tối ưu (8 x 8 = 64), giúp ổn định đạo hàm mà không cần code tay dài dòng.
- **Cơ chế Chống Overfitting**: Áp dụng Early Stopping (`patience=15`), kết hợp ngắt Augmentation (`close_mosaic=10`) ở các chu kỳ cuối nhằm tối ưu hóa độ sắc nét của viền đối tượng.

In [ ]:
from ultralytics import YOLO

# Tải kiến trúc mạng YOLO11n trọng số khởi tạo chuẩn
model = YOLO('yolo11n.pt')

# Khởi động quá trình Huấn luyện
results = model.train(
    data='d:/DAT301m/proposal/data/bdd100k.yaml',
    epochs=50,
    patience=15,           # Dừng sớm nếu sau 15 epoch mAP không tăng (Chống Overfit)
    batch=8,               # Kích thước khung nạp vật lý
    cache='disk',          # Bộ đệm ổ cứng giảm tải System RAM
    imgsz=640,
    device=0,
    optimizer='AdamW',     # Giải thuật tối ưu chuyên biệt cho tập nhiễu hạt
    mosaic=0.5,            # Bật Data Augmentation ghép 4 ảnh
    close_mosaic=10,       # Tắt ghép ảnh ở 10 epoch cuối để học viền thực tế
    project='d:/DAT301m/proposal/models/runs',
    name='bdd100k_train'
)